<a href="https://colab.research.google.com/github/EMADUDDINAsdaq/federated-learning-fairness-xray/blob/main/dissertation_22_06_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Federated Learning Aggregation Strategies for Client and Demographic Fairness in Chest X-Ray Classification
**Author:** Emaduddin Asdaq Syed Mohammed | **Module:** CSC8639 MSc Data Science and AI  
**Supervisor:** Jichun Li | **Newcastle University** | **Submission:** 10 August 2025

---

## Notebook Structure
| Cells | Section |
|---|---|
| 1–3 | Environment setup — GPU, Drive, Kaggle |
| 4–7 | Dataset download and image indexing |
| 8–15 | Metadata loading, exploration, cleaning |
| 16–18 | GPU optimisation and library installation |
| 19–23 | Dirichlet partition → 5 hospital clients |
| 24–28 | PyTorch transforms, Dataset class, model |
| 29 | Evaluation function (AUC + FNR per subgroup) |
| 30 | 10% sample subset for pipeline validation |
| 31–35 | **FedAvg** — McMahan et al. 2017 |
| 36–40 | **q-FedAvg** — Li et al. 2020 |
| 41–46 | **GIFAIR-FL** — Yue et al. 2022 |
| 47 | Final comparison table |

---

## Key Design Decisions
- **Framework:** Flower (flwr) — committed in interim report [Beutel et al. 2020]
- **Dataset:** NIH ChestX-ray14 via Kaggle (nih-chest-xrays/data Version 3)
- **Partition:** Dirichlet Dir(α=0.5) across 5 clients [Morafah et al. 2022]
- **Model:** ResNet-18 pretrained ImageNet [Zech et al. 2018]
- **Metrics:** AUC (performance) + FNR (underdiagnosis) per client and per demographic subgroup [Seyyed-Kalantari et al. 2021]


## Section 1 — Environment Setup

In [ ]:
pip install flwr protobuf


In [ ]:
!pip install "flwr[simulation]" protobuf -q
print("✓ Libraries installed")

✓ Libraries installed


In [ ]:
import flwr as fl
print(f"flwr : {fl.__version__}")
print("✓ Flower working")

flwr : 1.31.0
✓ Flower working


In [ ]:
# Import all libraries needed for the full dissertation notebook
import os, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
import flwr as fl
from flwr.common import (NDArrays, parameters_to_ndarrays,
                          ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
from typing import Dict, List, Optional, Tuple
warnings.filterwarnings('ignore')

print(f"flwr     : {fl.__version__}")
print(f"torch    : {torch.__version__}")
print(f"numpy    : {np.__version__}")
print(f"GPU      : {torch.cuda.is_available()}")
print("✓ All libraries ready")

flwr     : 1.31.0
torch    : 2.11.0+cu128
numpy    : 2.0.2
GPU      : True
✓ All libraries ready


In [ ]:
from google.colab import drive
import time, os
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import flwr.simulation
print("Simulation module loaded")

Simulation module loaded


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL 1 — Session initialisation
# Run this first every time the runtime restarts
# Confirms GPU is available before any training cells

import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=== Session Initialisation ===")
print(f"Device     : {device}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory     : {mem:.1f} GB")
    print(f"CUDA       : {torch.version.cuda}")
    print("\n✓ GPU ready — safe to run training cells")
else:
    print("\n⚠ No GPU detected")
    print("  Runtime → Change runtime type → T4 GPU (or A100 for full dataset)")
    print("  Then Runtime → Restart session → run this cell again")

=== Session Initialisation ===
Device     : cuda
GPU        : NVIDIA L4
Memory     : 23.7 GB
CUDA       : 12.8

✓ GPU ready — safe to run training cells


## Section 2 — Dataset Download and Image Indexing

In [ ]:
# CELL 4 — Download NIH ChestX-ray14 from Kaggle
# Dataset: nih-chest-xrays/data Version  (112,120 images, ~45 GB)
# Citation: Wang et al. 2017 — ChestX-ray8: Hospital-scale chest X-ray database
#           IEEE Conference on Computer Vision and Pattern Recognition (CVPR)
# NOTE: First download takes ~15–20 min on Colab. Subsequent runs use cache.

#import kagglehub

#DATASET_PATH = kagglehub.dataset_download("nih-chest-xrays/data")
#print(f"✓ Dataset path : {DATASET_PATH}")

In [ ]:
import os, time, zipfile

ZIP_PATH     = '/content/drive/MyDrive/dissertation/archive.zip'
EXTRACT      = '/content/nih_unzipped'
DATASET_PATH = EXTRACT

os.makedirs(EXTRACT, exist_ok=True)

# Get total files in zip for progress bar
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()
    total     = len(all_files)

print(f"ZIP contains : {total:,} files")
print(f"Extracting to: {EXTRACT}\n")

t0 = time.time()

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for i, file in enumerate(all_files):
        z.extract(file, EXTRACT)
        pct  = (i + 1) / total * 100
        bar  = '█' * int(pct / 2) + '││' * (50 - int(pct / 2))
        mins = (time.time() - t0) / 60
        print(f"\r[{bar}] {pct:5.1f}% | {i+1:,}/{total:,} | {mins:.1f} min",
              end='', flush=True)

elapsed = time.time() - t0
print(f"\n\n✓ Extraction complete in {elapsed/60:.1f} minutes")

# List top level contents
print(f"\n=== Contents of {EXTRACT} ===\n")
for item in sorted(os.listdir(EXTRACT)):
    p = os.path.join(EXTRACT, item)
    if os.path.isdir(p):
        n = len(os.listdir(p))
        print(f"  [DIR]  {item:<30} {n} items")
    else:
        print(f"  [FILE] {item:<30} {os.path.getsize(p)/1e6:.1f} MB")

ZIP contains : 112,128 files
Extracting to: /content/nih_unzipped

[██████████████████████████████████████████████████] 100.0% | 112,128/112,128 | 12.2 min

✓ Extraction complete in 12.2 minutes

=== Contents of /content/nih_unzipped ===

  [FILE] ARXIV_V5_CHESTXRAY.pdf         9.0 MB
  [FILE] BBox_List_2017.csv             0.1 MB
  [FILE] Data_Entry_2017.csv            7.9 MB
  [FILE] FAQ_CHESTXRAY.pdf              0.1 MB
  [FILE] LOG_CHESTXRAY.pdf              0.0 MB
  [FILE] README_CHESTXRAY.pdf           0.8 MB
  [DIR]  images_001                     1 items
  [DIR]  images_002                     1 items
  [DIR]  images_003                     1 items
  [DIR]  images_004                     1 items
  [DIR]  images_005                     1 items
  [DIR]  images_006                     1 items
  [DIR]  images_007                     1 items
  [DIR]  images_008                     1 items
  [DIR]  images_009                     1 items
  [DIR]  images_010                     1 items

In [ ]:
import os
for item in sorted(os.listdir('/content/nih_unzipped'))[:40]:
    print(item)

ARXIV_V5_CHESTXRAY.pdf
BBox_List_2017.csv
Data_Entry_2017.csv
FAQ_CHESTXRAY.pdf
LOG_CHESTXRAY.pdf
README_CHESTXRAY.pdf
images_001
images_002
images_003
images_004
images_005
images_006
images_007
images_008
images_009
images_010
images_011
images_012
test_list.txt
train_val_list.txt


In [ ]:
# Verify unzipped Drive dataset — top level contents
DATASET_PATH = '/content/nih_unzipped'

print("=== NIH ChestX-ray14 (Drive unzip) — Top Level Contents ===\n")
print(f"{'Name':<35} {'Type':<8} {'Size/Items':>12}")
print("-" * 58)
for item in sorted(os.listdir(DATASET_PATH)):
    p = os.path.join(DATASET_PATH, item)
    if os.path.isdir(p):
        print(f"{item:<35} {'DIR':<8} {len(os.listdir(p)):>10} items")
    else:
        print(f"{item:<35} {'FILE':<8} {os.path.getsize(p)/1e6:>9.1f} MB")

=== NIH ChestX-ray14 (Drive unzip) — Top Level Contents ===

Name                                Type       Size/Items
----------------------------------------------------------
ARXIV_V5_CHESTXRAY.pdf              FILE           9.0 MB
BBox_List_2017.csv                  FILE           0.1 MB
Data_Entry_2017.csv                 FILE           7.9 MB
FAQ_CHESTXRAY.pdf                   FILE           0.1 MB
LOG_CHESTXRAY.pdf                   FILE           0.0 MB
README_CHESTXRAY.pdf                FILE           0.8 MB
images_001                          DIR               1 items
images_002                          DIR               1 items
images_003                          DIR               1 items
images_004                          DIR               1 items
images_005                          DIR               1 items
images_006                          DIR               1 items
images_007                          DIR               1 items
images_008                          DIR 

In [ ]:
# CELL 7 — Build image path index — filename → full path dictionary
# Used by PyTorch Dataset to open each image during training
# One dict lookup per image — faster than os.walk at training time

print("Building image path index...")
image_path_dict = {}
for folder in sorted(os.listdir(DATASET_PATH)):
    if folder.startswith('images_'):
        img_dir = os.path.join(DATASET_PATH, folder, 'images')
        if os.path.isdir(img_dir):
            for f in os.listdir(img_dir):
                if f.endswith('.png'):
                    image_path_dict[f] = os.path.join(img_dir, f)
print(f"✓ Indexed {len(image_path_dict):,} images")
# Spot check
sample = '00000001_000.png'
if sample in image_path_dict:
    print(f"✓ Spot check OK : {image_path_dict[sample]}")

Building image path index...
✓ Indexed 112,120 images
✓ Spot check OK : /content/nih_unzipped/images_001/images/00000001_000.png


## Section 3 — Metadata Loading, Exploration and Cleaning

In [ ]:
# CELL 8 — Load raw metadata CSV exactly as provided by Kaggle
# Data_Entry_2017.csv — one row per image
# Columns: Image Index, Finding Labels, Patient Age, Patient Gender, ...

import pandas as pd

df_raw = pd.read_csv(f"{DATASET_PATH}/Data_Entry_2017.csv")
print(f"Rows    : {len(df_raw):,}")
print(f"Columns : {len(df_raw.columns)}")
print("\nColumn names:")
for i, col in enumerate(df_raw.columns):
    print(f"  {i+1:>2}. {col}")

Rows    : 112,120
Columns : 12

Column names:
   1. Image Index
   2. Finding Labels
   3. Follow-up #
   4. Patient ID
   5. Patient Age
   6. Patient Gender
   7. View Position
   8. OriginalImage[Width
   9. Height]
  10. OriginalImagePixelSpacing[x
  11. y]
  12. Unnamed: 11


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL 9 — Inspect data types and first rows

print("=== Data Types ===")
print(df_raw.dtypes)
print("\n=== First 3 Rows ===")
print(df_raw.head(3).to_string())

=== Data Types ===
Image Index                     object
Finding Labels                  object
Follow-up #                      int64
Patient ID                       int64
Patient Age                      int64
Patient Gender                  object
View Position                   object
OriginalImage[Width              int64
Height]                          int64
OriginalImagePixelSpacing[x    float64
y]                             float64
Unnamed: 11                    float64
dtype: object

=== First 3 Rows ===
        Image Index          Finding Labels  Follow-up #  Patient ID  Patient Age Patient Gender View Position  OriginalImage[Width  Height]  OriginalImagePixelSpacing[x     y]  Unnamed: 11
0  00000001_000.png            Cardiomegaly            0           1           58              M            PA                 2682     2749                        0.143  0.143          NaN
1  00000001_001.png  Cardiomegaly|Emphysema            1           1           58              M 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL 10 — Check missing values

print("=== Missing Values ===\n")
for col in df_raw.columns:
    n = df_raw[col].isnull().sum()
    status = '✓ none' if n == 0 else f'⚠  {n:,} missing'
    print(f"  {col:<35} {status}")

=== Missing Values ===

  Image Index                         ✓ none
  Finding Labels                      ✓ none
  Follow-up #                         ✓ none
  Patient ID                          ✓ none
  Patient Age                         ✓ none
  Patient Gender                      ✓ none
  View Position                       ✓ none
  OriginalImage[Width                 ✓ none
  Height]                             ✓ none
  OriginalImagePixelSpacing[x         ✓ none
  y]                                  ✓ none
  Unnamed: 11                         ⚠  112,120 missing


In [ ]:
# CELL 11 — Explore pathology label distribution
# 15 labels — images can have multiple labels separated by |
# 'No Finding' means no pathology detected

all_labels   = df_raw['Finding Labels'].str.split('|').explode()
label_counts = all_labels.value_counts()
print(f"{'Label':<25} {'Count':>8} {'%':>7}")
print("-" * 42)
for label, count in label_counts.items():
    print(f"{label:<25} {count:>8,} {count/len(df_raw)*100:>6.1f}%")
print(f"\nNo Finding rate : {label_counts['No Finding']/len(df_raw)*100:.1f}%")

Label                        Count       %
------------------------------------------
No Finding                  60,361   53.8%
Infiltration                19,894   17.7%
Effusion                    13,317   11.9%
Atelectasis                 11,559   10.3%
Nodule                       6,331    5.6%
Mass                         5,782    5.2%
Pneumothorax                 5,302    4.7%
Consolidation                4,667    4.2%
Pleural_Thickening           3,385    3.0%
Cardiomegaly                 2,776    2.5%
Emphysema                    2,516    2.2%
Edema                        2,303    2.1%
Fibrosis                     1,686    1.5%
Pneumonia                    1,431    1.3%
Hernia                         227    0.2%

No Finding rate : 53.8%


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL 12 — Explore sex distribution
# Note: original column is 'Patient Gender' — renamed during cleaning

sex = df_raw['Patient Gender'].value_counts()
pct = df_raw['Patient Gender'].value_counts(normalize=True).mul(100)
print(f"{'Sex':<10} {'Count':>8} {'%':>8}")
print("-" * 28)
for s in sex.index:
    print(f"{s:<10} {sex[s]:>8,} {pct[s]:>7.2f}%")

Sex           Count        %
----------------------------
M            63,340   56.49%
F            48,780   43.51%


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# CELL 13 — Explore age distribution — identify outliers

import numpy as np

print(f"Min age   : {df_raw['Patient Age'].min()}")
print(f"Max age   : {df_raw['Patient Age'].max()}")
print(f"Mean age  : {df_raw['Patient Age'].mean():.1f}")
print(f"Median age: {df_raw['Patient Age'].median():.0f}")
print()
above_100 = df_raw[df_raw['Patient Age'] > 100]
print(f"Rows with age > 100 : {len(above_100)}")
print("Values above 100 (data entry errors):")
print(above_100['Patient Age'].value_counts().sort_index())
print("\nConclusion: all ages > 100 are data entry errors — will be removed")

Min age   : 1
Max age   : 414
Mean age  : 46.9
Median age: 49

Rows with age > 100 : 16
Values above 100 (data entry errors):
Patient Age
148    2
149    1
150    1
151    1
152    1
153    1
154    1
155    2
411    1
412    3
413    1
414    1
Name: count, dtype: int64

Conclusion: all ages > 100 are data entry errors — will be removed


In [ ]:
# CELL 14 — Verify image filenames match metadata
# Sample 5 files from images_001 and check against metadata

folder_001   = os.path.join(DATASET_PATH, 'images_001', 'images')
sample_files = sorted(os.listdir(folder_001))[:5]
print(f"{'Filename':<25} {'In Metadata':>12} {'Finding Labels'}")
print("-" * 65)
for fname in sample_files:
    match = df_raw[df_raw['Image Index'] == fname]
    if len(match) > 0:
        print(f"{fname:<25} {'✓ YES':>12}  {match.iloc[0]['Finding Labels']}")
    else:
        print(f"{fname:<25} {'⚠ NOT FOUND':>12}")

Filename                   In Metadata Finding Labels
-----------------------------------------------------------------
00000001_000.png                 ✓ YES  Cardiomegaly
00000001_001.png                 ✓ YES  Cardiomegaly|Emphysema
00000001_002.png                 ✓ YES  Cardiomegaly|Effusion
00000002_000.png                 ✓ YES  No Finding
00000003_000.png                 ✓ YES  Hernia


In [ ]:
# CELL 15 — Clean metadata — 5 steps
# All steps documented in interim report Section 3.2
# Demographic groups aligned with Seyyed-Kalantari et al. 2021 (Nature Medicine)

df = df_raw.copy()

# Step 1 — rename Patient Gender to Patient Sex (neutral clinical terminology)
df = df.rename(columns={'Patient Gender': 'Patient Sex'})
print("✓ Step 1 — Patient Gender → Patient Sex")

# Step 2 — remove entirely null columns (Unnamed: 11 in Kaggle version)
null_cols = [c for c in df.columns if df[c].isnull().all()]
df        = df.drop(columns=null_cols)
print(f"✓ Step 2 — Removed null columns: {null_cols}")

# Step 3 — remove 16 age outliers above 100 (data entry errors — not clinical ages)
before = len(df)
df     = df[df['Patient Age'] <= 100]
print(f"✓ Step 3 — Removed {before - len(df)} rows (age > 100 — data entry errors)")
print(f"           Age range now: {df['Patient Age'].min()} – {df['Patient Age'].max()}")

# Step 4 — binary label: 1 = any pathology, 0 = No Finding
df['label'] = (df['Finding Labels'] != 'No Finding').astype(int)
print(f"✓ Step 4 — Binary label added | Pathology rate: {df['label'].mean()*100:.1f}%")

# Step 5 — age group bins for demographic subgroup evaluation
# Citation: Seyyed-Kalantari et al. 2021 used similar age group boundaries
df['Age Group'] = pd.cut(
    df['Patient Age'],
    bins   = [0, 20, 40, 60, 80, 101],
    labels = ['0-20', '20-40', '40-60', '60-80', '80+']
)
print("✓ Step 5 — Age groups: 0-20 | 20-40 | 40-60 | 60-80 | 80+")

# Add full image path to each row
df['image_path'] = df['Image Index'].map(image_path_dict)
matched   = df['image_path'].notna().sum()
unmatched = df['image_path'].isna().sum()
print(f"✓ Step 6 — Image paths linked: {matched:,} matched | {unmatched} unmatched")
print(f"\nFinal dataset  : {len(df):,} rows | {len(df.columns)} columns")

✓ Step 1 — Patient Gender → Patient Sex
✓ Step 2 — Removed null columns: ['Unnamed: 11']
✓ Step 3 — Removed 16 rows (age > 100 — data entry errors)
           Age range now: 1 – 95
✓ Step 4 — Binary label added | Pathology rate: 46.2%
✓ Step 5 — Age groups: 0-20 | 20-40 | 40-60 | 60-80 | 80+


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


✓ Step 6 — Image paths linked: 112,104 matched | 0 unmatched

Final dataset  : 112,104 rows | 14 columns


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Section 4 — Library Installation and GPU Optimisation

In [ ]:
# CELL 16 — Install required libraries
# flwr    — Flower federated learning framework
#           Citation: Beutel et al. 2020 arXiv:2007.14390 [ref 16 in interim report]
#           Committed in interim report: "A federated learning environment has been
#           set up using the Flower framework"
# torch   — PyTorch deep learning
# scikit-learn — AUC metric computation

import flwr as fl
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import roc_auc_score
from typing import Dict, List, Optional, Tuple
from flwr.common import (NDArrays, Scalar, Parameters,
                          parameters_to_ndarrays, ndarrays_to_parameters,
                          FitIns, FitRes, EvaluateIns, EvaluateRes)
from flwr.server.strategy import Strategy
from flwr.server.client_proxy import ClientProxy
import numpy as np
import warnings, json, os, time, copy
warnings.filterwarnings('ignore')

print(f"✓ Flower   : {fl.__version__}")
print(f"✓ PyTorch  : {torch.__version__}")
print(f"✓ GPU      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU name : {torch.cuda.get_device_name(0)}")

✓ Flower   : 1.31.0
✓ PyTorch  : 2.11.0+cu128
✓ GPU      : True
✓ GPU name : NVIDIA L4


In [ ]:
# CELL 17 — Confirm device after library install

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    print("\n✓ GPU confirmed — safe to proceed")
else:
    print("\n⚠ No GPU — go to Runtime → Change runtime type")

Device : cuda
GPU    : NVIDIA L4
Memory : 23.7 GB

✓ GPU confirmed — safe to proceed


In [ ]:
# CELL 18 — GPU optimisation settings
# Applied once here — used globally by all training cells
# BATCH_SIZE=256 for A100 | use 128 if on T4 (comment/uncomment below)
# persistent_workers=True — stops DataLoader workers restarting between epochs
# cudnn.benchmark=True — auto-selects fastest conv algorithm for fixed input size

torch.backends.cudnn.benchmark     = True   # faster conv for fixed image size
torch.backends.cudnn.deterministic = False  # allows non-deterministic speedups

BATCH_SIZE  = 512   # A100: 256 | T4: 128
NUM_WORKERS = 4     # parallel data loading processes
PREFETCH    = 2     # batches prefetched per worker

print(f"✓ BATCH_SIZE  : {BATCH_SIZE}  (use 128 on T4, 256 on A100)")
print(f"✓ NUM_WORKERS : {NUM_WORKERS}")
print(f"✓ PREFETCH    : {PREFETCH}")
print(f"✓ cuDNN bench : enabled")
print(f"✓ persistent_workers will be set on all DataLoaders")

✓ BATCH_SIZE  : 512  (use 128 on T4, 256 on A100)
✓ NUM_WORKERS : 4
✓ PREFETCH    : 2
✓ cuDNN bench : enabled
✓ persistent_workers will be set on all DataLoaders


## Section 5 — Dirichlet Partition → 5 Simulated Hospital Clients

In [ ]:
# CELL 19 — Dirichlet partition parameters
# Citation: Morafah et al. 2022 — "Rethinking Data Heterogeneity in FL"
#           IEEE Transactions on Artificial Intelligence
# "Dir(α=0.5) induces moderate-to-strong non-IID heterogeneity consistent
#  with real hospital data distributions" — Morafah et al. 2022
# seed=42 — fixed for reproducibility across all experiments

np.random.seed(42)

NUM_CLIENTS    = 5
ALPHA          = 0.5
HOSPITAL_NAMES = ['Hospital_A', 'Hospital_B', 'Hospital_C',
                  'Hospital_D', 'Hospital_E']

print(f"Clients  : {NUM_CLIENTS}")
print(f"Alpha    : {ALPHA}  — Dir(α=0.5) [Morafah et al. 2022]")
print(f"Seed     : 42")
print(f"Clients  : {HOSPITAL_NAMES}")

Clients  : 5
Alpha    : 0.5  — Dir(α=0.5) [Morafah et al. 2022]
Seed     : 42
Clients  : ['Hospital_A', 'Hospital_B', 'Hospital_C', 'Hospital_D', 'Hospital_E']


In [ ]:
# CELL 20 — Separate images by class label before Dirichlet sampling
# Dirichlet independently samples proportions for class 0 and class 1
# This creates label skew — each hospital sees a different class ratio

idx_0 = np.where(df['label'].values == 0)[0]
idx_1 = np.where(df['label'].values == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

print(f"Class 0 — No Finding : {len(idx_0):,} ({len(idx_0)/len(df)*100:.1f}%)")
print(f"Class 1 — Pathology  : {len(idx_1):,} ({len(idx_1)/len(df)*100:.1f}%)")
print(f"Total                : {len(df):,}")

Class 0 — No Finding : 60,353 (53.8%)
Class 1 — Pathology  : 51,751 (46.2%)
Total                : 112,104


In [ ]:
# CELL 21 — Sample Dirichlet proportions for each class independently
# Each hospital receives a different fraction of No Finding and Pathology images
# This is the source of the non-IID label skew

props_0 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))
props_1 = np.random.dirichlet(np.repeat(ALPHA, NUM_CLIENTS))

print(f"{'Client':<14} {'No Finding%':>12} {'Pathology%':>12}")
print("-" * 40)
for i, name in enumerate(HOSPITAL_NAMES):
    print(f"{name:<14} {props_0[i]*100:>11.1f}% {props_1[i]*100:>11.1f}%")

Client          No Finding%   Pathology%
----------------------------------------
Hospital_A            40.8%        71.2%
Hospital_B             0.0%        20.5%
Hospital_C            53.1%         5.6%
Hospital_D             0.8%         2.0%
Hospital_E             5.2%         0.7%


In [ ]:
# CELL 22 — Assign images to hospital clients using sampled proportions
# Last client absorbs rounding remainder to ensure all images assigned

splits_0 = (props_0 * len(idx_0)).astype(int)
splits_1 = (props_1 * len(idx_1)).astype(int)
splits_0[-1] = len(idx_0) - splits_0[:-1].sum()   # fix rounding
splits_1[-1] = len(idx_1) - splits_1[:-1].sum()

clients = {}
ptr0, ptr1 = 0, 0
for i, name in enumerate(HOSPITAL_NAMES):
    idx = np.concatenate([
        idx_0[ptr0 : ptr0 + splits_0[i]],
        idx_1[ptr1 : ptr1 + splits_1[i]]
    ])
    clients[name] = df.iloc[idx].copy().reset_index(drop=True)
    ptr0 += splits_0[i]
    ptr1 += splits_1[i]

assigned = sum(len(d) for d in clients.values())
print(f"Total assigned : {assigned:,}")
print(f"Total dataset  : {len(df):,}")
print(f"Match          : {'✓' if assigned == len(df) else '⚠ MISMATCH'}")

Total assigned : 112,104
Total dataset  : 112,104
Match          : ✓


In [ ]:
# CELL 23 — Print client statistics — verify non-IID distribution
# Expected: pathology rates differ strongly across hospitals
# This confirms Dirichlet partition is working as intended

print("=" * 80)
print(f"HOSPITAL CLIENT STATISTICS — Dirichlet Dir(α={ALPHA}) [Morafah et al. 2022]")
print("=" * 80)
print(f"\n{'Client':<14} {'Images':>8} {'Pathology%':>12} {'No Finding%':>13} "
      f"{'Age Mean':>9} {'Male%':>7}")
print("-" * 68)

for name, data in clients.items():
    n      = len(data)
    path   = data['label'].mean() * 100
    nof    = 100 - path
    age    = data['Patient Age'].mean()
    male   = (data['Patient Sex'] == 'M').mean() * 100
    print(f"{name:<14} {n:>8,} {path:>11.1f}% {nof:>12.1f}% {age:>8.1f} {male:>6.1f}%")

rates    = [d['label'].mean() for d in clients.values()]
variance = np.var(rates)
print("-" * 68)
print(f"\nPathology range across clients : {min(rates)*100:.1f}% – {max(rates)*100:.1f}%")
print(f"Variance across clients        : {variance:.6f}")
print(f"\n✓ Non-IID confirmed — strong label skew across hospital clients")
print(f"  This motivates fairness-aware aggregation strategies")

HOSPITAL CLIENT STATISTICS — Dirichlet Dir(α=0.5) [Morafah et al. 2022]

Client           Images   Pathology%   No Finding%  Age Mean   Male%
--------------------------------------------------------------------
Hospital_A       61,505        59.9%         40.1%     47.2   56.4%
Hospital_B       10,655        99.7%          0.3%     48.2   56.8%
Hospital_C       34,936         8.2%         91.8%     45.9   56.6%
Hospital_D        1,495        69.1%         30.9%     47.8   58.0%
Hospital_E        3,513        10.0%         90.0%     46.3   55.6%
--------------------------------------------------------------------

Pathology range across clients : 8.2% – 99.7%
Variance across clients        : 0.125535

✓ Non-IID confirmed — strong label skew across hospital clients
  This motivates fairness-aware aggregation strategies


## Section 6 — PyTorch Setup: Transforms, Dataset, Model

In [ ]:
# CELL 24 — Image preprocessing transforms
# NIH images: 1024×1024 grayscale PNG — must be resized and converted to RGB
# Resize 224×224 — standard input size for ImageNet pretrained models
# Grayscale→RGB — ResNet-18 expects 3 channels (repeats the grayscale channel)
# Normalise with ImageNet mean/std — standard practice for transfer learning
# Citation: Zech et al. 2018 used same preprocessing approach on NIH dataset
# Training augmentation: RandomHorizontalFlip (chest X-rays are L-R symmetric)
#                        RandomRotation(10°) — minor patient positioning variation

IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

print(f"✓ Image size    : {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"✓ Channels      : 3 (grayscale → RGB)")
print(f"✓ Normalisation : ImageNet mean/std [Zech et al. 2018]")
print(f"✓ Train augment : RandomHorizontalFlip + RandomRotation(10°)")
print(f"✓ Val augment   : none (consistent evaluation)")

✓ Image size    : 224×224
✓ Channels      : 3 (grayscale → RGB)
✓ Normalisation : ImageNet mean/std [Zech et al. 2018]
✓ Train augment : RandomHorizontalFlip + RandomRotation(10°)
✓ Val augment   : none (consistent evaluation)


In [ ]:
# CELL 25 — PyTorch Dataset class for chest X-ray images
# One instance per hospital client — wraps that client's dataframe
# Opens PNG from disk, converts to RGB, applies transforms
# Returns (image tensor [3,224,224], binary label float)

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

print("✓ ChestXrayDataset defined")
print("  Input  : dataframe row with image_path and label columns")
print("  Output : (tensor [3,224,224], float label)")

✓ ChestXrayDataset defined
  Input  : dataframe row with image_path and label columns
  Output : (tensor [3,224,224], float label)


In [ ]:
# CELL 26 — Verify one image loads correctly

test_ds    = ChestXrayDataset(clients['Hospital_A'], transform=val_transform)
image, lbl = test_ds[0]
print(f"Dataset size : {len(test_ds):,}")
print(f"Image shape  : {image.shape}")
print(f"Pixel range  : {image.min():.3f} – {image.max():.3f}")
print(f"Label        : {lbl.item()} ({'Pathology' if lbl.item()==1 else 'No Finding'})")
print("\n✓ Image loads correctly — pipeline ready")

Dataset size : 61,505
Image shape  : torch.Size([3, 224, 224])
Pixel range  : -2.118 – 2.413
Label        : 0.0 (No Finding)

✓ Image loads correctly — pipeline ready


In [ ]:
# CELL 27 — ResNet-18 model definition
# Pretrained on ImageNet — transfer learning for chest X-ray classification
# Final fully-connected layer replaced: 512 → 1 binary output
# BCEWithLogitsLoss — numerically stable binary cross-entropy (includes sigmoid)
# Adam lr=1e-4 — standard for fine-tuning pretrained convolutional networks
# Citation: Zech et al. 2018 used ResNet on NIH ChestX-ray14
ROUNDS = 20
#ROUNDS = 2
def build_model():
    model    = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(model.fc.in_features, 1)   # replace 512→1000 with 512→1
    return model

# Verify build
test_model = build_model()
params     = sum(p.numel() for p in test_model.parameters())
print(f"✓ ResNet-18 — ImageNet pretrained [Zech et al. 2018]")
print(f"✓ Final layer : 512 → 1 (binary classification)")
print(f"✓ Loss        : BCEWithLogitsLoss")
print(f"✓ Optimiser   : Adam lr=1e-4")
print(f"✓ Parameters  : {params:,}")
del test_model

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 214MB/s]


✓ ResNet-18 — ImageNet pretrained [Zech et al. 2018]
✓ Final layer : 512 → 1 (binary classification)
✓ Loss        : BCEWithLogitsLoss
✓ Optimiser   : Adam lr=1e-4
✓ Parameters  : 11,177,025


In [ ]:
# CELL 28 — 10% sample subset for pipeline validation
# Purpose: verify the full pipeline runs end-to-end before committing to
#          the full 112,104 image dataset (which takes ~5–8 hours on A100)
# Each client is sampled at 10% maintaining its label distribution
# Hospital_D and Hospital_E are very small — model may struggle — expected
# This is part of the fairness finding, not a pipeline error

SAMPLE_FRACTION = 0.10   # change to 1.0 for full dataset run on A100

sample_clients = {
    name: data.sample(frac=SAMPLE_FRACTION, random_state=42).reset_index(drop=True)
    for name, data in clients.items()
}

print(f"=== {'10%' if SAMPLE_FRACTION==0.10 else 'FULL'} Sample Subset ===\n")
print(f"{'Client':<14} {'Sample':>8} {'Full':>8} {'Pathology%':>12}")
print("-" * 46)
for name, data in sample_clients.items():
    print(f"{name:<14} {len(data):>8,} {len(clients[name]):>8,} "
          f"{data['label'].mean()*100:>11.1f}%")
print(f"\n✓ sample_clients ready — {'pipeline test' if SAMPLE_FRACTION==0.10 else 'FULL DATASET RUN'}")

=== 10% Sample Subset ===

Client           Sample     Full   Pathology%
----------------------------------------------
Hospital_A        6,150   61,505        60.2%
Hospital_B        1,066   10,655        99.8%
Hospital_C        3,494   34,936         8.5%
Hospital_D          150    1,495        69.3%
Hospital_E          351    3,513         9.1%

✓ sample_clients ready — pipeline test


## Section 7 — Evaluation Function

In [ ]:
# CELL 29 — Evaluation function — AUC and FNR per client and per demographic
#
# AUC  — Area Under ROC Curve — primary performance metric
#         Measures discriminative ability regardless of threshold
#
# FNR  — False Negative Rate = FN / (FN + TP)
#         Measures underdiagnosis — a false negative means pathology MISSED
#         Clinical consequence: patient sent home without treatment
#         Citation: Seyyed-Kalantari et al. 2021 Nature Medicine
#         "FNR is the critical metric for clinical underdiagnosis bias"
#
# Demographic subgroups evaluated:
#   Sex      : Male (M) vs Female (F)
#   Age group: 0-20 | 20-40 | 40-60 | 60-80 | 80+
#   Citation : Seyyed-Kalantari et al. 2021 | Ahluwalia et al. 2023
#
# Returns dict with keys:
#   auc, fnr, accuracy
#   auc_M, fnr_M, auc_F, fnr_F
#   auc_0-20, fnr_0-20, ... auc_80+, fnr_80+

def evaluate_client(model, dataframe, device):
    model = model.to(device)
    model.eval()

    loader = DataLoader(
        ChestXrayDataset(dataframe, transform=val_transform),
        batch_size         = BATCH_SIZE,
        shuffle            = False,
        num_workers        = NUM_WORKERS,
        pin_memory         = True,
        persistent_workers = True
    )

    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            probs = torch.sigmoid(
                model(images.to(device, non_blocking=True))
            ).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs > 0.5).astype(int)

    def auc_fnr_for_mask(y_true, y_prob, y_pred):
        if len(y_true) < 10 or y_true.sum() == 0:
            return float('nan'), float('nan')
        try:
            auc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auc = float('nan')
        fn  = int(((y_pred == 0) & (y_true == 1)).sum())
        tp  = int(((y_pred == 1) & (y_true == 1)).sum())
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
        return round(auc, 4), round(fnr, 4)

    # Overall metrics
    auc, fnr = auc_fnr_for_mask(labels, probs, preds)
    acc      = round(float((preds == labels).mean() * 100), 2)
    metrics  = {'auc': auc, 'fnr': fnr, 'accuracy': acc}

    # Per sex subgroup
    for sex in ['M', 'F']:
        mask = dataframe['Patient Sex'].values == sex
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{sex}'] = a
            metrics[f'fnr_{sex}'] = f

    # Per age group
    for grp in ['0-20', '20-40', '40-60', '60-80', '80+']:
        mask = dataframe['Age Group'].values == grp
        if mask.sum() > 10:
            a, f = auc_fnr_for_mask(labels[mask], probs[mask], preds[mask])
            metrics[f'auc_{grp}'] = a
            metrics[f'fnr_{grp}'] = f

    return metrics

print("✓ evaluate_client() defined")
print("  Metrics  : AUC + FNR (overall, per sex, per age group)")
print("  Citation : Seyyed-Kalantari et al. 2021 | Ahluwalia et al. 2023")

✓ evaluate_client() defined
  Metrics  : AUC + FNR (overall, per sex, per age group)
  Citation : Seyyed-Kalantari et al. 2021 | Ahluwalia et al. 2023


## Section 8 — Flower Client Base Class

All three FL methods share the same `HospitalClient` base for the standard `fit()` and `evaluate()` interface.  
GIFAIR-FL overrides `fit()` to accept a gradient scaling factor from the server.

**Flower framework** — Beutel et al. 2020 arXiv:2007.14390  
Interface: `get_parameters` | `set_parameters` | `fit` | `evaluate`


In [ ]:
# CELL 30 — Flower NumPyClient base — shared by FedAvg and q-FedAvg
#
# Flower simulation pattern (Beutel et al. 2020):
#   - Each hospital = one NumPyClient instance
#   - get_parameters() — return model weights as numpy arrays
#   - fit()            — receive global weights, train locally, return updated weights
#   - evaluate()       — receive global weights, evaluate on local data, return metrics
#   - Flower simulation runs all clients virtually on one GPU

class HospitalClient(fl.client.NumPyClient):
    """Standard Flower client — used for FedAvg and q-FedAvg."""

    def __init__(self, name: str, dataframe, device):
        self.name      = name
        self.dataframe = dataframe
        self.device    = device
        self.model     = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy()
                for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.dataframe, self.device)
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
               len(self.dataframe), m

print("✓ HospitalClient (Flower NumPyClient) defined")
print("  Used by: FedAvg | q-FedAvg")

✓ HospitalClient (Flower NumPyClient) defined
  Used by: FedAvg | q-FedAvg


## Section 9 — Method 1: FedAvg (McMahan et al. 2017)

**Algorithm 1 — McMahan et al. 2017 (exact):**
```
For each round t = 0 ... T-1:
  Server sends w_t to all clients
  Each client k:
    w_k^{t+1} ← run E epochs of SGD on local data F_k starting from w_t
  Server aggregates:
    w^{t+1} = Σ_k (n_k / n) · w_k^{t+1}
    where n_k = number of samples on client k, n = Σ n_k
```

**Flower's built-in `FedAvg` strategy implements this exactly.**  
No custom strategy needed — used as-is.

> H.B. McMahan et al. *Communication-efficient learning of deep networks from decentralized data.* AISTATS 2017.


In [ ]:
# CELL 31 — FedAvg client factory
# Flower simulation calls this function with cid = '0', '1', ... '4'
# Maps integer cid to hospital name and returns the correct client

def make_client_fn(client_data_map, device):
    def client_fn(cid: str) -> fl.client.Client:
        name = HOSPITAL_NAMES[int(cid)]
        return HospitalClient(
            name      = name,
            dataframe = client_data_map[name],
            device    = device
        ).to_client()
    return client_fn

print(f"✓ client_fn factory — maps cid 0–4 to {HOSPITAL_NAMES}")

✓ client_fn factory — maps cid 0–4 to ['Hospital_A', 'Hospital_B', 'Hospital_C', 'Hospital_D', 'Hospital_E']


In [ ]:
# CELL 32 — FedAvg strategy configuration
# Saves final parameters after each round for evaluation

import logging
logging.getLogger('flwr').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')

class FedAvgWithSave(fl.server.strategy.FedAvg):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.final_parameters = None

    def aggregate_fit(self, server_round, results, failures):
        aggregated = super().aggregate_fit(server_round, results, failures)
        if aggregated[0] is not None:
            self.final_parameters = aggregated[0]
        return aggregated

fedavg_strategy = FedAvgWithSave(
    fraction_fit          = 1.0,
    fraction_evaluate     = 1.0,
    min_fit_clients       = NUM_CLIENTS,
    min_evaluate_clients  = NUM_CLIENTS,
    min_available_clients = NUM_CLIENTS,
    on_fit_config_fn      = lambda rnd: {'epochs': 3, 'lr': 1e-4}
)
print("✓ FedAvg strategy ready [McMahan et al. 2017]")
print(f"  Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")

✓ FedAvg strategy ready [McMahan et al. 2017]
  Rounds: 20 | Epochs/round: 3 | Clients: 5


In [ ]:
# CELL 33 — Run FedAvg simulation

# Run FedAvg simulation — 10% sample

import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 50)
print("FedAvg — McMahan et al. 2017")
print(f"Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print("=" * 50)

t0 = time.time()

fedavg_history = fl.simulation.start_simulation(
    client_fn        = make_client_fn(sample_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = fedavg_strategy,
    client_resources = {'num_gpus': 1.0}
)

elapsed = time.time() - t0
print(f"\n{'='*50}")
print(f"✓ FedAvg complete in {elapsed/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in fedavg_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

FedAvg — McMahan et al. 2017
Rounds: 20 | Epochs/round: 3 | Clients: 5


2026-06-23 20:46:24,811	INFO worker.py:2012 -- Started a local Ray instance.
(ClientAppActor pid=17431) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=17431) 
(ClientAppActor pid=17431)             This is a deprecated feature. It will be removed
(ClientAppActor pid=17431)             entirely in future versions of Flower.
(ClientAppActor pid=17431)         
(ClientAppActor pid=17431) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=17431) 
(ClientAppActor pid=17431)             This is a deprecated feature. It will be removed
(ClientAppActor pid=17431)     


✓ FedAvg complete in 116.8 minutes

Loss per round:
  Round  1 : 0.2965
  Round  2 : 0.2621
  Round  3 : 0.2425
  Round  4 : 0.2324
  Round  5 : 0.2026
  Round  6 : 0.1744
  Round  7 : 0.1703
  Round  8 : 0.1601
  Round  9 : 0.1285
  Round 10 : 0.1364
  Round 11 : 0.1276
  Round 12 : 0.1123
  Round 13 : 0.0920
  Round 14 : 0.0865
  Round 15 : 0.0753
  Round 16 : 0.0890
  Round 17 : 0.0552
  Round 18 : 0.0626
  Round 19 : 0.0490
  Round 20 : 0.0497


In [ ]:
#CELL 34 — Evaluate FedAvg per client
fedavg_metrics = {}
final_params = parameters_to_ndarrays(fedavg_strategy.final_parameters)

for name, data in sample_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in final_params]
    )))
    fedavg_metrics[name] = evaluate_client(model, data, device)

# Main results table
rows = []
for name in HOSPITAL_NAMES:
    m = fedavg_metrics[name]
    rows.append({
        'Client'   : name,
        'AUC'      : m['auc'],
        'FNR'      : m['fnr'],
        'Accuracy' : m['accuracy'],
        'AUC_M'    : m.get('auc_M', 'N/A'),
        'FNR_M'    : m.get('fnr_M', 'N/A'),
        'AUC_F'    : m.get('auc_F', 'N/A'),
        'FNR_F'    : m.get('fnr_F', 'N/A'),
    })

df_results = pd.DataFrame(rows).set_index('Client')
print("=== FedAvg Results — McMahan et al. 2017 ===\n")
print(df_results.to_string())

auc_var = np.var([fedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES])
print(f"\nAUC Variance : {auc_var:.6f}")

=== FedAvg Results — McMahan et al. 2017 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.9957  0.3534     78.73  0.9955  0.3590  0.9960  0.3461
Hospital_B  0.7251  0.6335     36.77     NaN  0.6466  0.7422  0.6156
Hospital_C  0.9658  0.1385     93.99  0.9599  0.1344  0.9745  0.1455
Hospital_D  0.6482  0.6442     50.00  0.6541  0.6349  0.6508  0.6585
Hospital_E  0.8124  0.4688     86.04  0.7577  0.5238  0.8920  0.3636

AUC Variance : 0.018052


In [ ]:
# CELL 35 — Save FedAvg results to Drive

SAVE_DIR = '/content/drive/MyDrive/dissertation/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(f'{SAVE_DIR}/fedavg_10pct_metrics.json', 'w') as f:
    json.dump(fedavg_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in final_params])),
    f'{SAVE_DIR}/fedavg_10pct_model.pth'
)

print("✓ FedAvg metrics saved to Drive")
print("✓ FedAvg model saved to Drive")

✓ FedAvg metrics saved to Drive
✓ FedAvg model saved to Drive


## Section 10 — Method 2: q-FedAvg (Li et al. 2020)

**Algorithm 2 — Li et al. 2020 (exact pseudocode):**
```
Input: q, L (Lipschitz constant ≈ 1/lr), η (learning rate)
For each round t = 0 ... T-1:
  Server sends w_t to all clients
  Each client k:
    Run E epochs SGD from w_t  →  get w̄_k
    Compute:
      Δw_k  =  L · (w_t − w̄_k)                     ← scaled gradient difference
      Δ_k   =  F_k^q(w_t) · Δw_k                    ← loss-weighted gradient
      h_k   =  q · F_k^(q−1)(w_t) · ‖Δw_k‖²
                + L · F_k^q(w_t)                     ← Lipschitz constant estimate
    Send Δ_k and h_k to server
  Server updates:
    w_{t+1}  =  w_t  −  (Σ_k Δ_k) / (Σ_k h_k)
```

**Why this is different from FedAvg:**  
Clients with **higher loss** (struggling hospitals) produce larger Δ_k and smaller h_k,  
giving them **more influence** over the global model update — the opposite of FedAvg.

> T. Li et al. *Fair resource allocation in federated learning.* ICLR 2020. Algorithm 2, p.5.


In [ ]:
# CELL 36 — q-FedAvg client

class QFedAvgHospitalClient(HospitalClient):

    def fit(self, parameters, config):
        self.set_parameters(parameters)

        epochs    = int(config.get('epochs', 3))
        lr        = float(config.get('lr', 1e-4))
        criterion = nn.BCEWithLogitsLoss()

        # Pre-loss loader — num_workers=0, no persistent_workers
        pre_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size  = BATCH_SIZE,
            shuffle     = False,
            num_workers = 0,
            pin_memory  = False,
        )

        # F_k(w_t) before local training [Li et al. 2020 Algorithm 2]
        self.model.eval()
        loss_sum, n = 0.0, 0
        with torch.no_grad():
            for images, labels in pre_loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)
                loss_sum += criterion(self.model(images), labels).item() * len(labels)
                n        += len(labels)
        loss_before = loss_sum / n

        # Training loader — same as HospitalClient base
        train_loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        self.model.train()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)
        total_loss, total_samples = 0.0, 0

        for _ in range(epochs):
            for images, labels in train_loader:
                images  = images.to(self.device, non_blocking=True)
                labels  = labels.to(self.device, non_blocking=True).unsqueeze(1)
                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name,
             'loss_before': float(loss_before)}
        )

print("✓ QFedAvgHospitalClient defined")
print("  Computes F_k(w_t) before local training [Li et al. 2020 Algorithm 2]")

✓ QFedAvgHospitalClient defined
  Computes F_k(w_t) before local training [Li et al. 2020 Algorithm 2]


In [ ]:
# CELL 37 — q-FedAvg client factory

def make_qfedavg_client_fn(client_data_map, device):
    def client_fn(cid):
        name = HOSPITAL_NAMES[int(cid)]
        return QFedAvgHospitalClient(
            name      = name,
            dataframe = client_data_map[name],
            device    = device
        ).to_client()
    return client_fn

print("✓ q-FedAvg client factory defined")

✓ q-FedAvg client factory defined


In [ ]:
# CELL 38 — q-FedAvg strategy — Li et al. 2020 Algorithm 2
# Q_PARAM = 0.5, L_PARAM = 1.0 (pragmatic — L=1/lr=1e4 freezes model in 20 rounds)

Q_PARAM = 0.5
L_PARAM = 1.0

class QFedAvgStrategy(Strategy):

    def __init__(self, q=0.5, L=1.0, num_clients=5):
        super().__init__()
        self.q              = q
        self.L              = L
        self.num_clients    = num_clients
        self._global_params = None

    def initialize_parameters(self, client_manager):
        ndarrays            = [v.cpu().numpy() for v in build_model().state_dict().values()]
        self._global_params = ndarrays
        return ndarrays_to_parameters(ndarrays)

    def configure_fit(self, server_round, parameters, client_manager):
        self._global_params = parameters_to_ndarrays(parameters)
        ins     = FitIns(parameters, {'epochs': 3, 'lr': 1e-4})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def aggregate_fit(self, server_round, results, failures):
        if not results:
            return None, {}

        w_t       = self._global_params
        sum_Delta = None
        sum_h     = 0.0

        for _, fit_res in results:
            w_bar_k = parameters_to_ndarrays(fit_res.parameters)
            loss_k  = max(float(fit_res.metrics.get('loss_before', 1.0)), 1e-10)

            delta_w_k = [self.L * (w - wb) for w, wb in zip(w_t, w_bar_k)]
            Delta_k   = [(loss_k ** self.q) * dw for dw in delta_w_k]
            norm_sq   = float(sum(np.sum(dw ** 2) for dw in delta_w_k))
            h_k       = (self.q * (loss_k ** (self.q - 1)) * norm_sq
                         + self.L * (loss_k ** self.q))

            sum_Delta = Delta_k if sum_Delta is None else \
                        [sd + dk for sd, dk in zip(sum_Delta, Delta_k)]
            sum_h    += h_k

        if sum_h == 0.0 or sum_Delta is None:
            return ndarrays_to_parameters(w_t), {}

        w_new               = [w - sd / sum_h for w, sd in zip(w_t, sum_Delta)]
        self._global_params = w_new
        return ndarrays_to_parameters(w_new), {}

    def configure_evaluate(self, server_round, parameters, client_manager):
        ins     = EvaluateIns(parameters, {})
        sampled = client_manager.sample(num_clients=self.num_clients,
                                        min_num_clients=self.num_clients)
        return [(c, ins) for c in sampled]

    def aggregate_evaluate(self, server_round, results, failures):
        if not results:
            return None, {}
        total_loss = sum(r.loss * r.num_examples for _, r in results)
        total_n    = sum(r.num_examples for _, r in results)
        return total_loss / total_n, {}

    def evaluate(self, server_round, parameters):
        return None

print(f"✓ QFedAvgStrategy defined [Li et al. 2020]  q={Q_PARAM}  L={L_PARAM}")

✓ QFedAvgStrategy defined [Li et al. 2020]  q=0.5  L=1.0


In [ ]:
# CELL 39 — Run q-FedAvg simulation

import os, logging
os.environ['RAY_SILENT_MODE'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)


print("=" * 50)
print("q-FedAvg — Li et al. 2020")
print(f"Rounds: {ROUNDS} | Epochs/round: 3 | Clients: {NUM_CLIENTS}")
print(f"q={Q_PARAM}  L={L_PARAM}")
print("=" * 50)

t0               = time.time()
qfedavg_strategy = QFedAvgStrategy(q=Q_PARAM, L=L_PARAM, num_clients=NUM_CLIENTS)

qfedavg_history = fl.simulation.start_simulation(
    client_fn        = make_qfedavg_client_fn(sample_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = qfedavg_strategy,
    client_resources = {'num_gpus': 1.0}
)

print(f"\n{'='*50}")
print(f"✓ q-FedAvg complete in {(time.time()-t0)/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in qfedavg_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

q-FedAvg — Li et al. 2020
Rounds: 20 | Epochs/round: 3 | Clients: 5
q=0.5  L=1.0


2026-06-23 22:45:31,407	INFO worker.py:2012 -- Started a local Ray instance.
(ClientAppActor pid=53245) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=53245) 
(ClientAppActor pid=53245)             This is a deprecated feature. It will be removed
(ClientAppActor pid=53245)             entirely in future versions of Flower.
(ClientAppActor pid=53245)         
(ClientAppActor pid=53245) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=53245) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=53245)   self.pid = os.fork()
(ClientAppActor pid=53245) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Pa


✓ q-FedAvg complete in 200.2 minutes

Loss per round:
  Round  1 : 0.5637
  Round  2 : 0.5636
  Round  3 : 0.5635
  Round  4 : 0.5635
  Round  5 : 0.5635
  Round  6 : 0.5634
  Round  7 : 0.5634
  Round  8 : 0.5633
  Round  9 : 0.5632
  Round 10 : 0.5632
  Round 11 : 0.5631
  Round 12 : 0.5631
  Round 13 : 0.5630
  Round 14 : 0.5630
  Round 15 : 0.5629
  Round 16 : 0.5628
  Round 17 : 0.5629
  Round 18 : 0.5628
  Round 19 : 0.5628
  Round 20 : 0.5627


In [ ]:
# CELL 40 — Evaluate q-FedAvg per client

qfedavg_metrics      = {}
qfedavg_final_params = qfedavg_strategy._global_params

for name, data in sample_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in qfedavg_final_params]
    )))
    qfedavg_metrics[name] = evaluate_client(model, data, device)

rows = []
for name in HOSPITAL_NAMES:
    m = qfedavg_metrics[name]
    rows.append({'Client': name, 'AUC': m['auc'], 'FNR': m['fnr'],
                 'Accuracy': m['accuracy'],
                 'AUC_M': m.get('auc_M','N/A'), 'FNR_M': m.get('fnr_M','N/A'),
                 'AUC_F': m.get('auc_F','N/A'), 'FNR_F': m.get('fnr_F','N/A')})

df_q = pd.DataFrame(rows).set_index('Client')
print("=== q-FedAvg Results — Li et al. 2020 ===\n")
print(df_q.to_string())

fa_var  = np.var([fedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES])
qfa_var = np.var([qfedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES])
print(f"\nAUC Variance — FedAvg: {fa_var:.6f}  |  q-FedAvg: {qfa_var:.6f}")
if qfa_var < fa_var:
    print(f"✓ q-FedAvg reduced variance by {(fa_var-qfa_var)/fa_var*100:.1f}%")
else:
    print("⚠ Variance not reduced — expected with low rounds; dissertation finding")

=== q-FedAvg Results — Li et al. 2020 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.4580  0.7695     43.69  0.4467  0.8034  0.4714  0.7248
Hospital_B  0.1645  0.7857     21.39     NaN  0.8225  0.2178  0.7356
Hospital_C  0.4723  0.7872     71.89  0.4577  0.8387  0.5059  0.7000
Hospital_D  0.4824  0.7692     40.00  0.4696  0.8095  0.5089  0.7073
Hospital_E  0.5370  0.6875     72.93  0.5210  0.7143  0.5900  0.6364

AUC Variance — FedAvg: 0.018052  |  q-FedAvg: 0.017400
✓ q-FedAvg reduced variance by 3.6%


In [ ]:
# CELL 40 — Compare FedAvg vs q-FedAvg so far

print("=== Interim Comparison: FedAvg vs q-FedAvg (AUC) ===\n")
print(f"{'Client':<14} {'FedAvg AUC':>12} {'q-FedAvg AUC':>14} {'Change':>8}")
print("-" * 52)
for name in HOSPITAL_NAMES:
    fa  = fedavg_metrics.get(name, {}).get('auc', float('nan'))
    qfa = qfedavg_metrics.get(name, {}).get('auc', float('nan'))
    chg = qfa - fa if not (np.isnan(fa) or np.isnan(qfa)) else float('nan')
    arrow = '↑' if chg > 0 else '↓' if chg < 0 else '='
    print(f"{name:<14} {fa:>12.4f} {qfa:>14.4f} {arrow}{abs(chg):>6.4f}")

fa_var  = np.var([fedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES])
qfa_var = np.var([qfedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES])
print(f"\nAUC Variance — FedAvg: {fa_var:.6f} | q-FedAvg: {qfa_var:.6f}")
if qfa_var < fa_var:
    red = (fa_var - qfa_var) / fa_var * 100
    print(f"✓ q-FedAvg reduced AUC variance by {red:.1f}% — confirms Li et al. 2020")
else:
    print("⚠ q-FedAvg did not reduce variance — check q and L parameters")

=== Interim Comparison: FedAvg vs q-FedAvg (AUC) ===

Client           FedAvg AUC   q-FedAvg AUC   Change
----------------------------------------------------
Hospital_A           0.9957         0.4580 ↓0.5377
Hospital_B           0.7251         0.1645 ↓0.5606
Hospital_C           0.9658         0.4723 ↓0.4935
Hospital_D           0.6482         0.4824 ↓0.1658
Hospital_E           0.8124         0.5370 ↓0.2754

AUC Variance — FedAvg: 0.018052 | q-FedAvg: 0.017400
✓ q-FedAvg reduced AUC variance by 3.6% — confirms Li et al. 2020


In [ ]:
# CELL 41 — Save q-FedAvg results to Drive

with open(f'{SAVE_DIR}/qfedavg_10pct_metrics.json', 'w') as f:
    json.dump(qfedavg_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in qfedavg_final_params])),
    f'{SAVE_DIR}/qfedavg_10pct_model.pth'
)

print("✓ q-FedAvg metrics saved to Drive")
print("✓ q-FedAvg model saved to Drive")

✓ q-FedAvg metrics saved to Drive
✓ q-FedAvg model saved to Drive


## Section 11 — Method 3: GIFAIR-FL (Yue et al. 2022)

**Algorithm 2 — Yue et al. 2022 (exact pseudocode):**
```
Input: λ (regularisation), p_k = N_k/N (client size fraction)
Initialise: r_k^0 = 0 for all k

For each round c = 0 ... C-1:
  Server computes for each client k:
    r_k^c  =  Σ_{j≠k} |F_k(θ) − F_j(θ)|      ← loss spread penalty
    scale_k =  1  +  (λ / p_k·|A_sk|) · r_k^c  ← scaling factor broadcast to client

  Server broadcasts θ and scale_k to each client k

  Each client k trains with SCALED gradient:
    θ_k^{t+1}  =  θ_k^t  −  η · scale_k · ∇F_k(θ_k^t)
    ↑ Fairness is applied INSIDE local training, NOT in aggregation

  Server aggregation = PLAIN UNWEIGHTED AVERAGE:
    θ̄^c  =  (1/|S_c|) · Σ_{k ∈ S_c}  θ_k^{(c+1)E}

  Update r_k^{c+1} using new local losses from this round
```

**Why this is different from FedAvg and q-FedAvg:**  
- Fairness is in the **gradient step**, not the aggregation  
- Clients with high loss spread (outlier hospitals) train with a stronger gradient  
- Aggregation is a simple unweighted average — no reweighting at the server  
- |A_sk| = 1 for individual fairness (each hospital is its own group)

> X. Yue, M. Nouiehed, R. Al Kontar. *GIFAIR-FL: A Framework for Group and Individual Fairness in Federated Learning.* INFORMS Journal on Data Science, 2022. Algorithm 2.


In [ ]:
# CELL 41 — GIFAIR-FL client — fairness scaling applied inside local training
#
# Key difference from HospitalClient:
# fit() receives 'scale' from server config
# Multiplies all gradients by scale before optimizer.step()
# This implements:
#   θ_k^{t+1} = θ_k^t − η · scale_k · ∇F_k(θ_k^t)
# per Yue et al. 2022 Algorithm 2

LAM = 0.1   # λ = 0.1 — moderate group fairness [Yue et al. 2022]


class GIFAIRHospitalClient(fl.client.NumPyClient):
    """
    GIFAIR-FL client — Yue et al. 2022 Algorithm 2.
    Receives fairness scale from server, applies inside gradient step.
    """

    def __init__(self, name: str, dataframe, device):
        self.name      = name
        self.dataframe = dataframe
        self.device    = device
        self.model     = build_model().to(device)

    def get_parameters(self, config) -> NDArrays:
        return [v.cpu().numpy()
                for v in self.model.state_dict().values()]

    def set_parameters(self, parameters: NDArrays):
        state_dict = dict(zip(
            self.model.state_dict().keys(),
            [torch.tensor(p) for p in parameters]
        ))
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters: NDArrays, config: Dict) -> Tuple[NDArrays, int, Dict]:
        self.set_parameters(parameters)
        self.model.train()

        epochs = int(config.get('epochs', 3))
        lr     = float(config.get('lr', 1e-4))

        # ── Yue et al. 2022 Algorithm 2 ───────────────────────────────────
        # Server sends scale_k = 1 + (λ / p_k) · r_k^c
        # |A_sk| = 1 (individual fairness — each hospital is its own group)
        scale = float(config.get('scale', 1.0))
        # ──────────────────────────────────────────────────────────────────

        loader = DataLoader(
            ChestXrayDataset(self.dataframe, transform=train_transform),
            batch_size         = BATCH_SIZE,
            shuffle            = True,
            num_workers        = NUM_WORKERS,
            pin_memory         = True,
            persistent_workers = True,
            prefetch_factor    = PREFETCH
        )

        criterion = nn.BCEWithLogitsLoss()
        optimiser = torch.optim.Adam(self.model.parameters(), lr=lr)

        total_loss, total_samples = 0.0, 0
        for _ in range(epochs):
            for images, labels in loader:
                images = images.to(self.device, non_blocking=True)
                labels = labels.to(self.device, non_blocking=True).unsqueeze(1)

                outputs = self.model(images)
                loss    = criterion(outputs, labels)
                optimiser.zero_grad()
                loss.backward()

                # ── Yue et al. 2022 Algorithm 2 — scale gradients ─────────
                # θ_k^{t+1} = θ_k^t − η · scale · ∇F_k(θ_k^t)
                # Multiply all parameter gradients by the server-sent scale
                for param in self.model.parameters():
                    if param.grad is not None:
                        param.grad.data.mul_(scale)
                # ─────────────────────────────────────────────────────────

                optimiser.step()
                total_loss    += loss.item() * len(labels)
                total_samples += len(labels)

        avg_loss = total_loss / total_samples
        return (
            self.get_parameters(config={}),
            len(self.dataframe),
            {'loss': float(avg_loss), 'client_name': self.name}
        )

    def evaluate(self, parameters: NDArrays, config: Dict) -> Tuple[float, int, Dict]:
        self.set_parameters(parameters)
        m = evaluate_client(self.model, self.dataframe, self.device)
        return float(1.0 - (m['auc'] if not np.isnan(m['auc']) else 0.5)), \
               len(self.dataframe), m


print("✓ GIFAIRHospitalClient defined — Yue et al. 2022 Algorithm 2")
print("  Fairness: gradient scaled by server-sent factor INSIDE local training")
print("  Aggregation: plain unweighted average (handled by GIFAIRStrategy)")

✓ GIFAIRHospitalClient defined — Yue et al. 2022 Algorithm 2
  Fairness: gradient scaled by server-sent factor INSIDE local training
  Aggregation: plain unweighted average (handled by GIFAIRStrategy)


In [ ]:
# CELL 42 — GIFAIR-FL custom Flower strategy
#
# Server responsibilities (Yue et al. 2022 Algorithm 2):
#   1. Each round: compute r_k = Σ_{j≠k} |loss_k - loss_j|
#   2. Compute scale_k = 1 + (λ / p_k) * r_k  for each client k
#   3. Send scale_k to client k via fit config
#   4. Receive trained weights from all clients
#   5. Aggregate = plain unweighted average  (1/K) Σ θ_k
#   6. Update stored losses for next round

class GIFAIRStrategy(Strategy):
    """
    GIFAIR-FL-Global server strategy — Yue et al. 2022 Algorithm 2.

    Key design:
    - Computes r_k (loss spread penalty) each round
    - Sends scale factor to each client
    - Aggregation is a plain unweighted average
    """

    def __init__(self, lam: float, num_clients: int, client_sizes: Dict[str, int]):
        super().__init__()
        self.lam         = lam
        self.num_clients = num_clients
        # p_k = N_k / N_total — client sampling probability
        total    = sum(client_sizes.values())
        self.p_k = {name: size / total for name, size in client_sizes.items()}
        # Initialise losses — r_k^0 = 0 means scale = 1.0 for first round
        self.client_losses: Dict[str, float] = {n: 1.0 for n in client_sizes}
        self._global_params: Optional[NDArrays] = None

    def initialize_parameters(self, client_manager):
        ndarrays = [v.cpu().numpy()
                    for v in build_model().state_dict().values()]
        self._global_params = ndarrays
        return ndarrays_to_parameters(ndarrays)

    def _compute_scales(self) -> Dict[str, float]:
        """
        Yue et al. 2022 Algorithm 2:
          r_k   = Σ_{j≠k} |F_k - F_j|
          scale = 1 + (λ / p_k) · r_k
          |A_sk| = 1 for individual fairness
        """
        names  = list(self.client_losses.keys())
        losses = [self.client_losses[n] for n in names]
        scales = {}
        for i, name in enumerate(names):
            r_k = sum(np.sign(losses[i] - losses[j])
                    for j in range(len(losses)) if j != i)
            p_k          = self.p_k[name]
            scales[name] = 1.0 + (self.lam / (p_k * 1)) * r_k
        return scales

    def configure_fit(self, server_round, parameters, client_manager):
        """Send global model + per-client scale to each client."""
        self._global_params = parameters_to_ndarrays(parameters)
        scales  = self._compute_scales()
        sampled = client_manager.sample(
            num_clients=self.num_clients,
            min_num_clients=self.num_clients
        )
        fit_ins_list = []
        for proxy in sampled:
            cid_int = int(proxy.cid) % NUM_CLIENTS
            name    = HOSPITAL_NAMES[cid_int]
            config = {'epochs': 3, 'lr': 1e-4,
                      'scale': float(scales.get(name, 1.0)),
                      'server_round': server_round}
            fit_ins_list.append((proxy, FitIns(parameters, config)))
        return fit_ins_list

    def aggregate_fit(
        self,
        server_round: int,
        results: List[Tuple[ClientProxy, FitRes]],
        failures
    ) -> Tuple[Optional[Parameters], Dict]:
        """
        Yue et al. 2022 Algorithm 2 aggregation — PLAIN UNWEIGHTED AVERAGE:
          θ̄^c = (1/|S_c|) · Σ_{k ∈ S_c} θ_k^{(c+1)E}
        Note: fairness is in local training — aggregation is standard average.
        """
        if not results:
            return None, {}

        all_weights = [parameters_to_ndarrays(r.parameters)
                       for _, r in results]
        n     = len(all_weights)
        w_new = [
            sum(w[layer_idx] for w in all_weights) / n
            for layer_idx in range(len(all_weights[0]))
        ]

        # Update client losses for next round r_k computation
        for _, fit_res in results:
            cname = fit_res.metrics.get('client_name', '')
            if cname in self.client_losses:
                self.client_losses[cname] = float(
                    fit_res.metrics.get('loss', 1.0))

        self._global_params = w_new
        return ndarrays_to_parameters(w_new), {}

    def configure_evaluate(self, server_round, parameters, client_manager):
        ins     = EvaluateIns(parameters, {})
        sampled = client_manager.sample(
            num_clients=self.num_clients,
            min_num_clients=self.num_clients
        )
        return [(client, ins) for client in sampled]

    def aggregate_evaluate(self, server_round, results, failures):
        if not results:
            return None, {}
        total_loss    = sum(r.loss * r.num_examples for _, r in results)
        total_samples = sum(r.num_examples for _, r in results)
        return total_loss / total_samples, {}

    def evaluate(self, server_round, parameters):
        return None


print("✓ GIFAIRStrategy defined — Yue et al. 2022 Algorithm 2")
print(f"  λ = {LAM}")
print("  r_k   = Σ_{{j≠k}} |F_k − F_j|  (loss spread penalty)")
print("  scale = 1 + (λ/p_k) · r_k      (sent to each client)")
print("  Aggregation: plain unweighted average (1/K) Σ θ_k")

✓ GIFAIRStrategy defined — Yue et al. 2022 Algorithm 2
  λ = 0.1
  r_k   = Σ_{{j≠k}} |F_k − F_j|  (loss spread penalty)
  scale = 1 + (λ/p_k) · r_k      (sent to each client)
  Aggregation: plain unweighted average (1/K) Σ θ_k


In [ ]:
# CELL 43 — GIFAIR client factory

def make_gifair_client_fn(client_data_map, device):
    def client_fn(cid: str) -> fl.client.Client:
        name = HOSPITAL_NAMES[int(cid)]
        return GIFAIRHospitalClient(
            name      = name,
            dataframe = client_data_map[name],
            device    = device
        ).to_client()
    return client_fn

print(f"✓ GIFAIR client factory — maps cid 0–4 to {HOSPITAL_NAMES}")

✓ GIFAIR client factory — maps cid 0–4 to ['Hospital_A', 'Hospital_B', 'Hospital_C', 'Hospital_D', 'Hospital_E']


In [ ]:
# CELL 44 — Run GIFAIR-FL simulation

import os, logging
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['RAY_DISABLE_IMPORT_WARNING'] = '1'
logging.getLogger('flwr').setLevel(logging.ERROR)

print("=" * 60)
print("RUNNING GIFAIR-FL — Yue et al. 2022")
print(f"Rounds: {ROUNDS} | λ: {LAM} | Clients: {NUM_CLIENTS}")
print("=" * 60)

t0 = time.time()

client_sizes    = {name: len(data) for name, data in sample_clients.items()}
gifair_strategy = GIFAIRStrategy(
    lam          = LAM,
    num_clients  = NUM_CLIENTS,
    client_sizes = client_sizes
)

gifair_history = fl.simulation.start_simulation(
    client_fn        = make_gifair_client_fn(sample_clients, device),
    num_clients      = NUM_CLIENTS,
    config           = fl.server.ServerConfig(num_rounds=ROUNDS),
    strategy         = gifair_strategy,
    client_resources = {'num_gpus': 1.0}
)

print(f"\n✓ GIFAIR-FL complete in {(time.time()-t0)/60:.1f} minutes")
print(f"\nLoss per round:")
for rnd, loss in gifair_history.losses_distributed:
    print(f"  Round {rnd:>2} : {loss:.4f}")

RUNNING GIFAIR-FL — Yue et al. 2022
Rounds: 20 | λ: 0.1 | Clients: 5


2026-06-24 02:07:10,796	INFO worker.py:2012 -- Started a local Ray instance.
(ClientAppActor pid=111395) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signature: {'cid': <Parameter "cid: str">}. You can import the `Context` like this: `from flwr.app import Context`
(ClientAppActor pid=111395) 
(ClientAppActor pid=111395)             This is a deprecated feature. It will be removed
(ClientAppActor pid=111395)             entirely in future versions of Flower.
(ClientAppActor pid=111395)         
(ClientAppActor pid=111395) /usr/lib/python3.12/multiprocessing/popen_fork.py:66: DeprecationWarning: This process (pid=111395) is multi-threaded, use of fork() may lead to deadlocks in the child.
(ClientAppActor pid=111395)   self.pid = os.fork()
(ClientAppActor pid=111395) WARNING :   DEPRECATED FEATURE: `client_fn` now expects a signature `def client_fn(context: Context)`.The provided `client_fn` has signatur


✓ GIFAIR-FL complete in 116.9 minutes

Loss per round:
  Round  1 : 0.3572
  Round  2 : 0.3614
  Round  3 : 0.3468
  Round  4 : 0.3385
  Round  5 : 0.3788
  Round  6 : 0.3191
  Round  7 : 0.3851
  Round  8 : 0.3846
  Round  9 : 0.3489
  Round 10 : 0.4057
  Round 11 : 0.4010
  Round 12 : 0.3767
  Round 13 : 0.4317
  Round 14 : 0.4464
  Round 15 : 0.4265
  Round 16 : 0.4608
  Round 17 : 0.4894
  Round 18 : 0.4848
  Round 19 : 0.5006
  Round 20 : 0.5103


In [ ]:
# CELL 45 — Evaluate GIFAIR-FL per client

gifair_metrics      = {}
gifair_final_params = gifair_strategy._global_params

for name, data in sample_clients.items():
    model = build_model().to(device)
    model.load_state_dict(dict(zip(
        model.state_dict().keys(),
        [torch.tensor(p) for p in gifair_final_params]
    )))
    gifair_metrics[name] = evaluate_client(model, data, device)

import pandas as pd
rows = []
for name in HOSPITAL_NAMES:
    m = gifair_metrics[name]
    rows.append({
        'Client'  : name,
        'AUC'     : m['auc'],
        'FNR'     : m['fnr'],
        'Accuracy': m['accuracy'],
        'AUC_M'   : m.get('auc_M', 'N/A'),
        'FNR_M'   : m.get('fnr_M', 'N/A'),
        'AUC_F'   : m.get('auc_F', 'N/A'),
        'FNR_F'   : m.get('fnr_F', 'N/A'),
    })

df_g = pd.DataFrame(rows).set_index('Client')
print("=== GIFAIR-FL Results — Yue et al. 2022 ===\n")
print(df_g.to_string())
auc_var = np.var([gifair_metrics[n]['auc'] for n in HOSPITAL_NAMES])
print(f"\nAUC Variance : {auc_var:.6f}")

=== GIFAIR-FL Results — Yue et al. 2022 ===

               AUC     FNR  Accuracy   AUC_M   FNR_M   AUC_F   FNR_F
Client                                                              
Hospital_A  0.5163  0.6706     49.27  0.5064  0.6486  0.5297  0.6997
Hospital_B  0.6325  0.7218     27.95     NaN  0.6873  0.6011  0.7689
Hospital_C  0.3884  0.7770     65.40  0.3748  0.7849  0.4049  0.7636
Hospital_D  0.5778  0.6442     48.67  0.5701  0.6667  0.5876  0.6098
Hospital_E  0.5583  0.6562     69.80  0.5392  0.6667  0.5894  0.6364

AUC Variance : 0.006744


In [ ]:
with open(f'{SAVE_DIR}/gifair_10pct_metrics.json', 'w') as f:
    json.dump(gifair_metrics, f, indent=2, default=str)

torch.save(
    dict(zip(build_model().state_dict().keys(),
             [torch.tensor(v) for v in gifair_final_params])),
    f'{SAVE_DIR}/gifair_10pct_model.pth'
)

print("✓ GIFAIR metrics saved")
print("✓ GIFAIR model saved")

✓ GIFAIR metrics saved
✓ GIFAIR model saved


## Section 12 — Final Comparison Table

All three methods compared on:
- **Inter-client fairness**: AUC per hospital + AUC variance across hospitals
- **Demographic fairness**: FNR gap between Male and Female patients

Citations: McMahan 2017 | Li 2020 | Yue 2022 | Seyyed-Kalantari 2021


In [ ]:
# CELL 46 — Final comparison table — main dissertation result
#
# Inter-client fairness metric  : AUC variance across 5 hospital clients
#   Lower variance = fairer performance distribution [Li et al. 2020]
#
# Demographic fairness metric   : FNR gap between Male and Female patients
#   Smaller gap = fairer clinical underdiagnosis [Seyyed-Kalantari et al. 2021]

print("=" * 75)
print("FINAL COMPARISON — FedAvg vs q-FedAvg vs GIFAIR-FL")
print("NIH ChestX-ray14 | 10% sample | 10 rounds | 3 epochs/round")
print("McMahan 2017 | Li 2020 | Yue 2022")
print("=" * 75)

all_results = {
    'FedAvg'  : fedavg_metrics,
    'q-FedAvg': qfedavg_metrics,
    'GIFAIR'  : gifair_metrics,
}

# ── Table 1: AUC per hospital client ──────────────────────────────────────
print(f"\n── AUC per Hospital Client ──────────────────────────────────────")
print(f"{'Client':<14}", end="")
for m in all_results:
    print(f" {m:>12}", end="")
print()
print("-" * 54)
for name in HOSPITAL_NAMES:
    print(f"{name:<14}", end="")
    for method, res in all_results.items():
        auc = res.get(name, {}).get('auc', float('nan'))
        print(f" {auc:>12.4f}", end="")
    print()

# ── Table 2: AUC variance ─────────────────────────────────────────────────
print(f"\n── Inter-Client AUC Variance (lower = fairer) ───────────────────")
print(f"{'Method':<12} {'Variance':>10} {'vs FedAvg':>12}")
print("-" * 36)
base_var = np.var([fedavg_metrics[n]['auc'] for n in HOSPITAL_NAMES])
for method, res in all_results.items():
    aucs = [res.get(n, {}).get('auc', float('nan')) for n in HOSPITAL_NAMES]
    var  = np.var([a for a in aucs if not np.isnan(a)])
    chg  = ((base_var - var) / base_var * 100) if method != 'FedAvg' else 0
    sign = f"{chg:+.1f}%" if method != 'FedAvg' else "baseline"
    print(f"{method:<12} {var:>10.6f} {sign:>12}")

# ── Table 3: FNR sex gap ──────────────────────────────────────────────────
print(f"\n── Demographic Fairness: FNR by Sex (lower gap = fairer) ────────")
print(f"{'Method':<12} {'FNR_M':>8} {'FNR_F':>8} {'Gap':>8}")
print("-" * 40)
for method, res in all_results.items():
    fnr_m_vals = [res.get(n,{}).get('fnr_M', np.nan) for n in HOSPITAL_NAMES]
    fnr_f_vals = [res.get(n,{}).get('fnr_F', np.nan) for n in HOSPITAL_NAMES]
    fnr_m = np.nanmean(fnr_m_vals)
    fnr_f = np.nanmean(fnr_f_vals)
    gap   = abs(fnr_m - fnr_f)
    print(f"{method:<12} {fnr_m:>8.4f} {fnr_f:>8.4f} {gap:>8.4f}")

# ── Summary ───────────────────────────────────────────────────────────────
print(f"\n── Summary ──────────────────────────────────────────────────────")
print("  FedAvg   : baseline — size-weighted average [McMahan et al. 2017]")
print("  q-FedAvg : gradient reweighting by loss^q — targets client fairness")
print("             [Li et al. 2020 Algorithm 2]")
print("  GIFAIR   : gradient scaling by loss spread — targets group fairness")
print("             [Yue et al. 2022 Algorithm 2]")
print()
print("  AUC variance  → inter-client performance fairness [Li et al. 2020]")
print("  FNR sex gap   → demographic underdiagnosis fairness [Seyyed-Kalantari 2021]")
print()
print("✓ Results ready for dissertation Chapter 4 (Results and Discussion)")
print(f"✓ All metrics saved to {SAVE_DIR}")

FINAL COMPARISON — FedAvg vs q-FedAvg vs GIFAIR-FL
NIH ChestX-ray14 | 10% sample | 10 rounds | 3 epochs/round
McMahan 2017 | Li 2020 | Yue 2022

── AUC per Hospital Client ──────────────────────────────────────
Client               FedAvg     q-FedAvg       GIFAIR
------------------------------------------------------
Hospital_A           0.9957       0.4580       0.5163
Hospital_B           0.7251       0.1645       0.6325
Hospital_C           0.9658       0.4723       0.3884
Hospital_D           0.6482       0.4824       0.5778
Hospital_E           0.8124       0.5370       0.5583

── Inter-Client AUC Variance (lower = fairer) ───────────────────
Method         Variance    vs FedAvg
------------------------------------
FedAvg         0.018052     baseline
q-FedAvg       0.017400        +3.6%
GIFAIR         0.006744       +62.6%

── Demographic Fairness: FNR by Sex (lower gap = fairer) ────────
Method          FNR_M    FNR_F      Gap
----------------------------------------
FedAvg    